# T04 — Muon + MOFA+: Multi-Modal Single-Cell Analysis

**Tools:** `muon`, `mofapy2`  
**Dataset:** PBMC CITE-seq (RNA + surface protein, 10x Genomics)  
**Papers:** [Bredikhin et al. 2022, Genome Biology](https://doi.org/10.1186/s13059-021-02577-8) · [Argelaguet et al. 2020, Genome Biology](https://doi.org/10.1186/s13059-020-02015-1)

---

## What is multi-modal single-cell data?

Modern protocols measure multiple molecular layers in the **same cell**:

| Protocol | Modalities |
|----------|------------|
| CITE-seq | RNA + surface protein (ADT) |
| 10x Multiome | RNA + ATAC (chromatin accessibility) |
| REAP-seq | RNA + protein |
| scNMT-seq | RNA + ATAC + methylation |
| Perturb-CITE-seq | RNA + protein + CRISPR guides |

## MuData: the multi-modal container

`MuData` is to `AnnData` what a multi-experiment object is to a count matrix. It stores multiple AnnData objects (modalities) in one file with:
- **Per-modality** obs and var metadata
- **Global** obs metadata (across modalities)
- Inter-modality linkages

```
MuData  n_obs × (n_vars_rna + n_vars_adt)
  ├── mod['rna']   AnnData  n_obs × n_vars_rna
  │     .X, .obs, .var, .obsm, ...
  └── mod['adt']   AnnData  n_obs × n_vars_adt
        .X, .obs, .var, .obsm, ...
```

## MOFA+ (Multi-Omics Factor Analysis)

MOFA+ is an unsupervised factor analysis model that finds **latent factors** shared across modalities. Like PCA, but:
- Works on multiple data matrices simultaneously
- Handles missing modalities per cell/sample
- Each factor can be active in some modalities and inactive in others
- Can be extended to temporal/spatial data (MEFISTO)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import muon as mu
from muon import prot as pt
import mofapy2
import matplotlib.pyplot as plt

sc.settings.set_figure_params(dpi=80, facecolor='white')
print(f'muon {mu.__version__}')

## 1. Load CITE-seq Data

We use the 10k PBMC CITE-seq dataset from 10x Genomics — RNA + 14 surface proteins.
Muon can read 10x HDF5 output directly.

In [ ]:
# Option A: use muon's built-in dataset loader
# This downloads ~150 MB PBMC CITE-seq data on first run
try:
    mdata = mu.datasets.pbmc5k()
    print('Loaded from muon datasets')
except AttributeError:
    # Option B: download from 10x and load manually
    import pooch
    url = 'https://cf.10xgenomics.com/samples/cell-exp/6.0.0/SC3_v3_NextGem_DI_PBMC_CSP_10K/SC3_v3_NextGem_DI_PBMC_CSP_10K_filtered_feature_bc_matrix.h5'
    path = pooch.retrieve(url, known_hash=None, fname='pbmc_cite_10k.h5')
    mdata = mu.read_10x_h5(path)
    print('Loaded from 10x h5')

print(mdata)
print('\nModalities:', list(mdata.mod.keys()))

In [ ]:
# Inspect each modality
rna = mdata['rna']
prot = mdata['prot']   # 'adt' in some versions
print('RNA:', rna)
print('\nProtein panel:', prot.var_names.tolist())

## 2. Per-Modality QC and Preprocessing

Each modality gets its own appropriate normalization:
- **RNA**: library-size normalization → log1p (same as T00)
- **Protein (ADT)**: Centered Log-Ratio (CLR) normalization — handles the very different count scale of antibody data

In [ ]:
# --- RNA preprocessing ---
rna = mdata['rna']
sc.pp.filter_cells(rna, min_genes=200)
sc.pp.filter_genes(rna, min_cells=10)
rna.var['mt'] = rna.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(rna, qc_vars=['mt'], inplace=True)
print('RNA QC done:', rna.shape)

In [ ]:
# Cell-level QC filters (applied to global mdata.obs via muon)
mu.pp.filter_obs(mdata, 'rna:n_genes_by_counts', lambda x: (x >= 200) & (x < 5000))
mu.pp.filter_obs(mdata, 'rna:pct_counts_mt', lambda x: x < 20)
print('After QC filtering:', mdata.shape)

In [ ]:
# RNA normalization
rna = mdata['rna']
rna.layers['counts'] = rna.X.copy()
sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)
sc.pp.highly_variable_genes(rna, n_top_genes=3000)
print('HVGs:', rna.var.highly_variable.sum())

In [ ]:
# --- Protein (ADT) normalization ---
# CLR: log( x_i / geometric_mean(x) ) — normalizes across proteins per cell
# OR across cells per protein; 'axis=1' = across proteins (recommended for CITE-seq)
prot = mdata['prot']
mu.prot.pp.clr(prot)  # Centered log-ratio, in-place
print('Protein CLR-normalized:', prot.X[:3, :5])

In [ ]:
# PCA per modality
sc.tl.pca(rna)
sc.tl.pca(prot)

# Propagate global obs to mdata
mu.pp.intersect_obs(mdata)
print('Final mdata:', mdata)

## 3. Independent Analysis Per Modality

Let's first cluster on RNA alone (as a baseline) before multi-modal integration.

In [ ]:
# RNA-only clustering
rna_hvg = rna[:, rna.var.highly_variable].copy()
sc.pp.scale(rna_hvg, max_value=10)
sc.tl.pca(rna_hvg)
sc.pp.neighbors(rna_hvg, n_pcs=30)
sc.tl.umap(rna_hvg)
sc.tl.leiden(rna_hvg, resolution=0.5, key_added='leiden_rna')

# Copy results back to the slice in mdata
rna.obsm['X_pca'] = rna_hvg.obsm['X_pca']
rna.obsm['X_umap'] = rna_hvg.obsm['X_umap']
rna.obs['leiden_rna'] = rna_hvg.obs['leiden_rna']

sc.pl.umap(rna_hvg, color='leiden_rna', legend_loc='on data', title='RNA-only clustering')

In [ ]:
# Plot a few protein markers on the RNA UMAP
known_proteins = [c for c in ['CD3E', 'CD4', 'CD8A', 'CD14', 'CD19', 'CD56']
                  if c in prot.var_names]
if known_proteins:
    # temporarily add protein data to rna_hvg.obs for plotting
    prot_df = pd.DataFrame(
        prot.X.toarray() if hasattr(prot.X, 'toarray') else prot.X,
        index=prot.obs_names, columns=prot.var_names
    )
    for p in known_proteins[:4]:
        if p in prot_df.columns:
            rna_hvg.obs[f'prot_{p}'] = prot_df.reindex(rna_hvg.obs_names)[p]
    sc.pl.umap(rna_hvg, color=[f'prot_{p}' for p in known_proteins[:4]], ncols=4)

## 4. MOFA+ Factor Analysis

MOFA+ finds shared latent factors that explain variation across both RNA and protein. Factors can be interpreted as:
- Cell cycle
- Cell type gradients
- Activation states
- Batch effects (if present)

In [ ]:
# MOFA+ via muon (convenience wrapper around mofapy2)
# n_factors: number of latent factors to learn (like number of PCs)
# use_obs='intersection' aligns cells across modalities

# Subset to HVGs for speed
mdata_mofa = mdata.copy()
mdata_mofa['rna'] = mdata_mofa['rna'][:, mdata_mofa['rna'].var.highly_variable]

mu.tl.mofa(
    mdata_mofa,
    n_factors=15,
    use_var='highly_variable',  # uses .var['highly_variable'] for RNA
    convergence_mode='fast',
    outfile='/tmp/mofa_pbmc.hdf5',  # saves trained model
    seed=42,
)
print('MOFA+ factors stored in mdata_mofa.obsm["X_mofa"]')
print('Shape:', mdata_mofa.obsm['X_mofa'].shape)

In [ ]:
# Variance explained per factor per modality
# This is the key diagnostic plot: what does each factor capture?
mu.pl.mofa_var_explained(mdata_mofa, figsize=(8, 4))

In [ ]:
# UMAP from MOFA factors (multi-modal embedding)
sc.pp.neighbors(mdata_mofa, use_rep='X_mofa', n_neighbors=20)
sc.tl.umap(mdata_mofa)
sc.tl.leiden(mdata_mofa, resolution=0.5, key_added='leiden_mofa')

sc.pl.umap(mdata_mofa, color='leiden_mofa', legend_loc='on data',
           title='MOFA+ multi-modal UMAP')

In [ ]:
# Top gene/protein loadings per factor
# Factors with high protein loadings are usually cell-type defining
mu.pl.mofa_loadings(mdata_mofa, factor=0, view='rna', n_features=15, figsize=(5, 6))

In [ ]:
mu.pl.mofa_loadings(mdata_mofa, factor=0, view='prot', n_features=14, figsize=(5, 6))

## 5. Weighted Nearest Neighbor (WNN) Embedding

Muon implements the Weighted Nearest Neighbor approach from Seurat v4: for each cell, compute a weighted combination of kNN graphs from each modality, where the weight reflects how informative that modality is for that cell. This tends to give better cluster resolution than either modality alone.

In [ ]:
# WNN requires PCA for each modality first (already done)
# Internally computes per-cell modality weights
mu.pp.neighbors(
    mdata,
    key_added='wnn',
    n_neighbors=20,
)

sc.tl.umap(mdata, neighbors_key='wnn')
sc.tl.leiden(mdata, neighbors_key='wnn', resolution=0.5, key_added='leiden_wnn')

sc.pl.umap(mdata, color='leiden_wnn', legend_loc='on data',
           title='WNN multi-modal UMAP')

In [ ]:
# MuData saves to .h5mu (HDF5-based multi-modal format)
mdata.write('../data/pbmc_cite_processed.h5mu')
print('Saved to ../data/pbmc_cite_processed.h5mu')

# Load back
# mdata2 = mu.read('../data/pbmc_cite_processed.h5mu')

---
## Summary

### MuData API pattern
```python
mdata = mu.read_10x_h5('file.h5')         # or mu.datasets.*
rna  = mdata['rna']                        # access modality as AnnData
prot = mdata['prot']

# Normalize per modality
sc.pp.normalize_total(rna); sc.pp.log1p(rna)
mu.prot.pp.clr(prot)

# Multi-modal analysis
mu.pp.intersect_obs(mdata)                 # align cells
mu.tl.mofa(mdata, n_factors=15)            # MOFA+ integration
mu.pp.neighbors(mdata, key_added='wnn')    # WNN graph

# Persistence
mdata.write('out.h5mu')
```

### MOFA+ interpretation
| Factor property | Interpretation |
|----------------|----------------|
| High variance explained in both modalities | Strong multi-modal signal (cell type) |
| High variance in RNA only | RNA-specific program |
| High variance in protein only | Likely surface marker gradient |
| Correlated with library size | Technical factor — inspect/regress |

**Done!** You've covered the full Theis lab / scverse ecosystem:
- **T00** AnnData + Scanpy (foundation)
- **T01** scvi-tools (deep generative models, integration)
- **T02** Squidpy (spatial transcriptomics)
- **T03** CellRank 2 (trajectory inference)
- **T04** Muon + MOFA+ (multi-modal analysis)